# **Preparation of the Data**

In [ ]:
import pandas as pd

# Cleaning ESC50 Dataset

In [ ]:
#Loading the entire ESC50 dataset in pandas
df = pd.read_csv("/content/esc50.csv")
df.head()

,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A
2,1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A
3,1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B
4,1-101296-A-19.wav,1,19,thunderstorm,False,101296,A


In [ ]:
#Filtering the needed sound lables from the dataframe. (These labels were manually selected from the dataset as per need)
labels = ["door_wood_knock", "church_bells", "clock_alarm", "glass_breaking", "siren"]

df = df[df['category'].isin(labels)]

print(df['category'].value_counts())

category
door_wood_knock    40
church_bells       40
clock_alarm        40
glass_breaking     40
siren              40
Name: count, dtype: int64


In [ ]:
df = df.reset_index(drop=True)

In [ ]:
#Exporting this dataframe in form of csv. This is the shorter version of the dataset that is to be used for our specific task
df.to_csv("Cleaned_ESC50.csv", index=False)

# Cleaning FSD50K Dataset

## Cleaning the dev dataset

In [ ]:
#loading the FSD50K dev data
clean_dev = pd.read_csv("/content/dev.csv")
clean_dev.info()
print(clean_dev['labels'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40966 entries, 0 to 40965
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   fname   40966 non-null  int64 
 1   labels  40966 non-null  object
 2   mids    40966 non-null  object
 3   split   40966 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.3+ MB
labels
Wind_instrument_and_woodwind_instrument,Musical_instrument,Music                       2543
Bowed_string_instrument,Musical_instrument,Music                                       1921
Piano,Keyboard_(musical),Musical_instrument,Music                                       766
Laughter,Human_voice                                                                    716
Snare_drum,Drum,Percussion,Musical_instrument,Music                                     707
                                                                                       ... 
Tearing,Wood                                                       

In [ ]:
#Filtering the needed sound lables from the dataframe. (These labels were manually selected from the dataset as per need)
val = ["Alarm", "Bell", "church_bell", "Doorbell", "Knock", "Siren"]

Carried out the filtering of all those audios which had even one label in our `val` list of needed audios. This is carried out through `re` (reguar expression) library

In [ ]:
"""As one audiofile corresponds to multiple lables, this code keeps all those audios
which has atleast one labels of our need in the dataframe"""

import re

pattern = '|'.join([re.escape(s) for s in val])
print(pattern,'\n')

clean_dev = clean_dev[clean_dev['labels'].str.contains(pattern, case=False, na=False)]

print("DataFrame filtered by content (partial matches):")
print(clean_dev['labels'].value_counts())

Alarm|Bell|church_bell|Doorbell|Knock|Siren 

DataFrame filtered by content (partial matches):
labels
Telephone,Alarm                                                                                                                            415
Alarm                                                                                                                                      314
Bell                                                                                                                                       220
Knock,Door,Domestic_sounds_and_home_sounds                                                                                                 181
Cowbell,Bell                                                                                                                               162
                                                                                                                                          ... 
Church_bell,Singing,Human_voice,Bell    

Now after filtering there are still some labels in the dataset that are not needed for that particular audio. For removing them we implemented `clean_label_string()` that does the needed cleaning.

In [ ]:
#Carried out this function for the cleaning of labels of interested data
def clean_label_string(label_string):
    if pd.isna(label_string):
        return None

    individual_labels_lower = [label.strip().lower() for label in label_string.split(',')]

    filtered_labels_exact_case = [
        original_val for original_val in val
        if original_val.lower() in individual_labels_lower
    ]

    return ','.join(filtered_labels_exact_case) if filtered_labels_exact_case else None

clean_dev_cleaned = clean_dev.copy()
clean_dev_cleaned['labels'] = clean_dev_cleaned['labels'].apply(clean_label_string)

print(clean_dev_cleaned['labels'].value_counts())

display(clean_dev_cleaned[['labels']].head())

labels
Alarm                     1016
Bell                       623
Knock                      270
Alarm,Doorbell              92
Bell,church_bell            80
Alarm,Bell                  79
Alarm,Siren                 77
Alarm,Bell,Doorbell         15
Alarm,Bell,church_bell       1
Name: count, dtype: int64


,labels
593,Bell
904,Alarm
937,Alarm
943,Alarm
944,Alarm


Now among the needed labels, All the labels are mapped to two needed/important categories which are **Alarm** and **Doorbell**.

In [ ]:
#Carried out this function to map all the labels to 'Alarm' or 'Doorbell'
def relabel_categories(label_string):
    if pd.isna(label_string):
        return None

    if ('alarm' in label_string.lower() and 'siren' in label_string.lower()) or (label_string.lower() == 'alarm'):
        return 'Alarm'
    else:
        return 'Doorbell'

clean_dev_cleaned['labels'] = clean_dev_cleaned['labels'].apply(relabel_categories)

print("Value counts after re-labeling:")
print(clean_dev_cleaned['labels'].value_counts())

display(clean_dev_cleaned[['labels']].head())

Value counts after re-labeling:
labels
Doorbell    1160
Alarm       1093
Name: count, dtype: int64


,labels
593,Doorbell
904,Alarm
937,Alarm
943,Alarm
944,Alarm


In [ ]:
clean_dev_cleaned.to_csv("FSD50K_dev_cleaned.csv", index=False)

## Cleaning the Evaluation Dataset

In [ ]:
clean_eval = pd.read_csv("/content/eval.csv")

In [ ]:
#checking for the same pattern we did for the dev dataset
clean_eval = clean_eval[clean_eval['labels'].str.contains(pattern, case=False, na=False)]

print(clean_eval['labels'].value_counts())

labels
Telephone,Alarm                                                                70
Alarm                                                                          55
Knock,Door,Domestic_sounds_and_home_sounds                                     52
Cowbell,Bell                                                                   45
Bicycle_bell,Bicycle,Vehicle,Alarm,Bell                                        36
                                                                               ..
Screech,Sliding_door,Alarm,Door,Domestic_sounds_and_home_sounds                 1
Walk_and_footsteps,Alarm                                                        1
Tap,Alarm                                                                       1
Alarm,Truck,Ocean,Boat_and_Water_vehicle,Motor_vehicle_(road),Vehicle,Water     1
Fixed-wing_aircraft_and_airplane,Alarm,Engine,Aircraft,Vehicle                  1
Name: count, Length: 357, dtype: int64


Applied the `clean_label_string()` function implemented before to clean the extra labels in each audio data points

In [ ]:
#Removing the extra labels in each label string

clean_eval_cleaned = clean_eval.copy()
clean_eval_cleaned['labels'] = clean_eval_cleaned['labels'].apply(clean_label_string)

print(clean_eval_cleaned['labels'].value_counts())

labels
Alarm                     430
Bell                      258
Knock                     103
Alarm,Bell                 61
Bell,church_bell           59
Alarm,Siren                54
Alarm,Doorbell             31
Alarm,Bell,Doorbell         6
Alarm,Bell,Siren            1
Alarm,Bell,church_bell      1
Name: count, dtype: int64


Relabeling every labels to 2 categories: **Alarm** and **Doorbell**

In [ ]:
#Mapping each label string to either Alarm or Doorbell
clean_eval_cleaned['labels'] = clean_eval_cleaned['labels'].apply(relabel_categories)

print(clean_eval_cleaned['labels'].value_counts())

labels
Doorbell    519
Alarm       485
Name: count, dtype: int64


In [ ]:
clean_eval_cleaned.to_csv("FSDK50_eval_cleaned.csv", index=False)